###Inicialización del Entorno y Estructura SCD2

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- 1. Dimensión Clientes (SCD2)
CREATE TABLE IF NOT EXISTS workspace.gold.dim_clientes (
    cliente_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    cliente_id INT,
    nombre_cliente STRING,
    ciudad STRING,
    valido_desde DATE,
    valido_hasta DATE,
    es_actual BOOLEAN
) USING DELTA;

-- 2. Dimensión Productos 
CREATE TABLE IF NOT EXISTS workspace.gold.dim_productos (
    producto_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    nombre_producto STRING,
    categoria STRING,
    precio_referencia DOUBLE
) USING DELTA;

###Lógica de Carga Incremental SCD2

In [0]:
%sql
-- Carga de Clientes (SCD2)
INSERT INTO workspace.gold.dim_clientes (cliente_id, nombre_cliente, ciudad, valido_desde, valido_hasta, es_actual)
SELECT cliente_id, nombre, ciudad, CURRENT_DATE(), CAST('9999-12-31' AS DATE), TRUE
FROM workspace.silver.clientes
WHERE cliente_id NOT IN (SELECT cliente_id FROM workspace.gold.dim_clientes WHERE es_actual = TRUE);

-- Carga de Productos
INSERT INTO workspace.gold.dim_productos (nombre_producto, precio_referencia)
SELECT DISTINCT producto, AVG(stock_disponible) 
FROM workspace.silver.inventario
WHERE producto NOT IN (SELECT nombre_producto FROM workspace.gold.dim_productos)
GROUP BY producto;

###Construcción y Carga de la Tabla de Hechos

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.fact_ventas (
    venta_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    cliente_sk BIGINT,
    producto_sk BIGINT, -- Nueva FK
    fecha_venta DATE,
    pais STRING,
    cantidad INT,
    valor_unitario DOUBLE,
    valor_total DOUBLE
) USING DELTA;

-- Carga cruzando con dimensiones
INSERT INTO workspace.gold.fact_ventas (cliente_sk, producto_sk, fecha_venta, pais, cantidad, valor_unitario, valor_total)
SELECT 
    c.cliente_sk,
    p.producto_sk,
    CAST(s.fecha_venta AS DATE),
    s.pais,
    s.cantidad,
    s.valor_unitario,
    s.valor_total
FROM workspace.silver.ventas s
JOIN workspace.gold.dim_clientes c ON s.cliente_id = c.cliente_id AND c.es_actual = TRUE
JOIN workspace.gold.dim_productos p ON s.producto = p.nombre_producto
WHERE CAST(s.fecha_venta AS DATE) > (SELECT COALESCE(MAX(fecha_venta), '1900-01-01') FROM workspace.gold.fact_ventas);

In [0]:

%sql
-- 1. Vista para identificar los productos estrella por cantidad física vendida
CREATE OR REPLACE VIEW workspace.gold.vw_top_5_productos_volumen AS
SELECT 
    p.nombre_producto,
    SUM(f.cantidad) AS unidades_vendidas,
    ROUND(SUM(f.valor_total), 2) AS ingresos_generados
FROM workspace.gold.fact_ventas f
JOIN workspace.gold.dim_productos p ON f.producto_sk = p.producto_sk
GROUP BY p.nombre_producto
ORDER BY unidades_vendidas DESC
LIMIT 5;


-- 2. Vista de distribución geográfica de ingresos
CREATE OR REPLACE VIEW workspace.gold.vw_ventas_por_pais AS
SELECT 
    pais,
    COUNT(venta_sk) AS transacciones_totales,
    SUM(cantidad) AS volumen_total,
    ROUND(SUM(f.valor_total), 2) AS facturacion_total
FROM workspace.gold.fact_ventas f
GROUP BY pais
ORDER BY facturacion_total DESC;



-- 3. Vista para identificar clientes VIP 
CREATE OR REPLACE VIEW workspace.gold.vw_clientes_vip AS
SELECT 
    c.nombre_cliente,
    c.ciudad,
    COUNT(f.venta_sk) AS frecuencia_compra,
    SUM(f.cantidad) AS total_productos_comprados,
    ROUND(SUM(f.valor_total), 2) AS valor_comprado_acumulado
FROM workspace.gold.fact_ventas f
JOIN workspace.gold.dim_clientes c ON f.cliente_sk = c.cliente_sk
WHERE c.es_actual = TRUE 
GROUP BY c.nombre_cliente, c.ciudad
ORDER BY valor_comprado_acumulado DESC;